# Connecting vision to LLMs: hands-on (optional)
**LLMs From Basics · Lesson MM.1**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

You will (1) count visual tokens for real resolutions and patch sizes, then (2) do LLaVA's **stage 1** in miniature: train a linear projector so frozen “image features” land next to the frozen embeddings of their caption words.

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab). Everything runs in a few seconds on a CPU.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

## 1 · Counting visual tokens
A ViT cuts an $H \times W$ image into $p \times p$ patches, so a LLaVA-style model sees $N = (H/p)(W/p)$ visual tokens. Some models (Qwen2-VL) then **merge each 2×2 block** of neighbouring patches into one token; if a side has an odd number of patches, the last row/column is padded, so each side contributes $\lceil n/2 \rceil$ tokens.

Implement `visual_tokens(h, w, patch, merge2x2=False)`. Assume `h` and `w` are multiples of `patch`.

Then implement `tiled_tokens(h, w, tile=336, patch=14)` for LLaVA-NeXT-style tiling: cover the image with a grid of `tile × tile` crops ($\lceil h/\text{tile} \rceil \times \lceil w/\text{tile} \rceil$ of them), encode each crop **plus one downscaled overview** of the whole image, each costing `visual_tokens(tile, tile, patch)`.

In [ ]:
def visual_tokens(h, w, patch, merge2x2=False):
    # TODO: patches per side, then (optionally) merge 2x2 blocks with ceil
    raise NotImplementedError("Exercise 1a")

def tiled_tokens(h, w, tile=336, patch=14):
    # TODO: number of tiles + 1 overview, each costing visual_tokens(tile, tile, patch)
    raise NotImplementedError("Exercise 1b")

**Check 1**

In [ ]:
assert visual_tokens(336, 336, 14) == 576, "LLaVA-1.5: 24 x 24"
assert visual_tokens(224, 224, 14) == 256
assert visual_tokens(448, 448, 14, merge2x2=True) == 256
assert visual_tokens(224, 224, 14, merge2x2=True) == 64, "Qwen2-VL: 64 (+2 start/end markers = 66)"
assert visual_tokens(336, 336, 16, merge2x2=True) == 121, "21 patches per side -> 11 after padding"
assert tiled_tokens(672, 672) == 2880, "LLaVA-NeXT: 2x2 tiles + overview = 5 x 576"
assert tiled_tokens(1008, 672) == 4032
print("✓ exercise 1")

## 2 · What share of the context?
Write `context_share(n_tokens, context)` (a fraction between 0 and 1, or above 1 if it doesn't fit) and `images_that_fit(n_tokens, context)` (how many such images fit, ignoring the text).

In [ ]:
def context_share(n_tokens, context):
    # TODO
    raise NotImplementedError("Exercise 2a")

def images_that_fit(n_tokens, context):
    # TODO
    raise NotImplementedError("Exercise 2b")

**Check 2** (also prints a small table)

In [ ]:
assert abs(context_share(576, 4096) - 0.140625) < 1e-12
assert images_that_fit(576, 4096) == 7
assert images_that_fit(tiled_tokens(672, 672), 4096) == 1
assert context_share(visual_tokens(1344, 1344, 14), 8192) > 1, "9,216 tokens don't fit in 8K"
print("✓ exercise 2")

print(f"{'image':>11} {'patch':>5} {'merge':>5} {'tokens':>7} {'% of 32K':>8}")
for side in (224, 336, 448, 672, 896, 1344):
    for p, m in ((14, False), (14, True)):
        n = visual_tokens(side, side, p, m)
        print(f"{side:>5}x{side:<5} {p:>5} {('2x2' if m else '-'):>5} {n:>7} {100 * context_share(n, 32768):>7.1f}%")

## Toy stage 1: the data
We can't load CLIP and an LLM here, so we fake both, keeping what matters:

- a **frozen word-embedding table** `E` (8 caption words, 16 dimensions), standing in for the LLM's input embeddings;
- **frozen image features** `X` (32 dimensions), standing in for a vision encoder's output. Each image shows one of the 8 objects, so its features are that object's (hidden) visual prototype plus noise.

The two spaces have different sizes and nothing lines them up. The projector's job is to learn a map $h = xW + b$ from image space (32-d) into word space (16-d) so that each image lands near its caption word.

In [ ]:
WORDS = ["cat", "dog", "car", "tree", "boat", "pizza", "clock", "bird"]
D_IMG, D_TXT, NOISE = 32, 16, 0.6

E = rng.normal(size=(len(WORDS), D_TXT))
E /= np.linalg.norm(E, axis=1, keepdims=True)          # frozen "LLM" word embeddings
PROTO = rng.normal(size=(len(WORDS), D_IMG))           # hidden visual prototypes

def make_images(n, noise=NOISE, rng=rng):
    labels = rng.integers(0, len(WORDS), size=n)
    X = PROTO[labels] + noise * rng.normal(size=(n, D_IMG))   # frozen "vision encoder" features
    return X, labels

X_train, y_train = make_images(400)
X_test, y_test = make_images(200)
T_train = E[y_train]          # target: the caption word's embedding
print(X_train.shape, T_train.shape, "first captions:", [WORDS[i] for i in y_train[:5]])

## 3 · The projector and its loss
Implement the projector `project(X, W, b)` $= XW + b$ for a whole batch, and the loss: the mean over images of the squared distance between the projected feature and its target word embedding,

$$\mathcal{L}(W, b) = \frac{1}{N}\sum_{i=1}^{N} \big\lVert x_i W + b - e_{y_i} \big\rVert^2$$

(LLaVA uses the next-token loss on the caption instead; a squared distance to the word's embedding keeps this toy small while asking for the same thing: put the image where its words are.)

In [ ]:
def project(X, W, b):
    # TODO: (N, D_IMG) @ (D_IMG, D_TXT) + (D_TXT,)
    raise NotImplementedError("Exercise 3a")

def mse_loss(H, T):
    # TODO: mean over rows of the squared Euclidean distance
    raise NotImplementedError("Exercise 3b")

**Check 3**

In [ ]:
W0, b0 = np.zeros((D_IMG, D_TXT)), np.zeros(D_TXT)
H0 = project(X_train, W0, b0)
assert H0.shape == (400, D_TXT)
assert abs(mse_loss(H0, T_train) - 1.0) < 1e-9, "all-zero projection: distance² to a unit vector is 1"
assert mse_loss(T_train, T_train) == 0
print("✓ exercise 3")

## 4 · Train the projector (stage 1)
Everything else is frozen: only $W$ and $b$ learn. The gradients of the loss are

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{2}{N} X^\top (H - T), \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{2}{N}\sum_i (h_i - t_i)$$

with $H = XW + b$. Implement `train_projector` with plain gradient descent (P2.2) and return the weights and the loss history.

In [ ]:
def train_projector(X, T, lr=0.01, steps=500, seed=1):
    r = np.random.default_rng(seed)
    W = 0.01 * r.normal(size=(X.shape[1], T.shape[1]))
    b = np.zeros(T.shape[1])
    losses = []
    for _ in range(steps):
        # TODO: forward (project), record the loss, compute gradients, take a step
        raise NotImplementedError("Exercise 4")
    return W, b, losses

**Check 4**

In [ ]:
W, b, losses = train_projector(X_train, T_train)
print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}")
assert len(losses) == 500
assert losses[-1] < 0.25 * losses[0], "the loss should drop a lot"
assert all(np.diff(losses[:50]) <= 1e-9), "with this learning rate the loss should go down every step"
plt.figure(figsize=(6, 3)); plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss"); plt.title("stage 1: only the projector learns"); plt.show()
print("✓ exercise 4")

## 5 · Did the images land next to their words?
For each projected test image, find the **nearest word embedding** by cosine similarity and compare it with the true caption word. Implement `nearest_word(H, E)` returning one word index per row, then compare an untrained projector with your trained one.

In [ ]:
def nearest_word(H, E):
    # TODO: cosine similarity between every row of H and every row of E, then argmax per row
    raise NotImplementedError("Exercise 5")

**Check 5**

In [ ]:
assert (nearest_word(E, E) == np.arange(len(WORDS))).all()
W_rand = rng.normal(size=(D_IMG, D_TXT)); b_rand = np.zeros(D_TXT)
acc_rand = np.mean(nearest_word(project(X_test, W_rand, b_rand), E) == y_test)
acc = np.mean(nearest_word(project(X_test, W, b), E) == y_test)
print(f"test accuracy: random projector {acc_rand:.2f}, trained projector {acc:.2f} (chance = {1/len(WORDS):.2f})")
assert acc >= 0.95, "the trained projector should map held-out images next to their words"
assert acc_rand < 0.5
print("✓ exercise 5")

## Experiment
1. **How much paired data does alignment need?** Train on 8, 16, 32, … images and plot test accuracy. (Real stage-1 runs use hundreds of thousands of pairs because the real map is far from linear and the concepts are millions, not 8.)
2. **Noisier features.** Re-run with `make_images(..., noise=1.5)`. Which degrades more, the loss or the accuracy?
3. **Too few dimensions.** Try `D_TXT = 4` in the data cell. Can 8 words still be told apart?

In [ ]:
sizes = [8, 16, 32, 64, 128, 256, 400]
accs = []
for n in sizes:
    Wn, bn, _ = train_projector(X_train[:n], T_train[:n], steps=400)
    accs.append(np.mean(nearest_word(project(X_test, Wn, bn), E) == y_test))
plt.figure(figsize=(6, 3)); plt.semilogx(sizes, accs, "o-", base=2)
plt.xlabel("training pairs"); plt.ylabel("test accuracy"); plt.ylim(0, 1.05); plt.title("alignment needs paired data"); plt.show()
print(dict(zip(sizes, np.round(accs, 2))))

## Recap
- Visual tokens grow with the **area** of the image: $(H/p)(W/p)$, divided by 4 if 2×2 patches are merged; tiling adds one tile's worth per crop plus an overview.
- Stage 1 trains **only the projector**: frozen image features are mapped into the frozen LLM's embedding space, next to the words that describe them.
- Once aligned, images can be fed to the LLM as tokens it can already “read”; stage 2 (instruction tuning) then teaches it to converse about them.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def visual_tokens(h, w, patch, merge2x2=False):
    nh, nw = h // patch, w // patch
    if merge2x2:
        return math.ceil(nh / 2) * math.ceil(nw / 2)
    return nh * nw

def tiled_tokens(h, w, tile=336, patch=14):
    tiles = math.ceil(h / tile) * math.ceil(w / tile)
    return (tiles + 1) * visual_tokens(tile, tile, patch)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def context_share(n_tokens, context):
    return n_tokens / context

def images_that_fit(n_tokens, context):
    return context // n_tokens

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def project(X, W, b):
    return X @ W + b

def mse_loss(H, T):
    return np.mean(np.sum((H - T) ** 2, axis=1))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def train_projector(X, T, lr=0.01, steps=500, seed=1):
    r = np.random.default_rng(seed)
    W = 0.01 * r.normal(size=(X.shape[1], T.shape[1]))
    b = np.zeros(T.shape[1])
    losses = []
    N = len(X)
    for _ in range(steps):
        H = project(X, W, b)
        losses.append(mse_loss(H, T))
        G = 2 * (H - T) / N
        W -= lr * (X.T @ G)
        b -= lr * G.sum(axis=0)
    return W, b, losses

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def nearest_word(H, E):
    Hn = H / np.linalg.norm(H, axis=1, keepdims=True)
    En = E / np.linalg.norm(E, axis=1, keepdims=True)
    return np.argmax(Hn @ En.T, axis=1)